# Customer Segmentation Analysis using RFM and K-Means

**Objective:** Segment an e-commerce customer base by purchasing behaviour and translate the segments into actionable marketing strategies.

**Dataset:** `OnlineRetail.csv` (Online Retail transactions)  
**Tools:** Python, pandas, NumPy, scikit-learn, matplotlib, seaborn, Jupyter Notebook

## Workflow
1. Load and inspect the data.
2. Clean invalid and incomplete transaction records.
3. Build customer-level **Recency, Frequency, Monetary (RFM)** features.
4. Log-transform skewed features and standardise with `StandardScaler`.
5. Use the Elbow Method and silhouette score to select a practical cluster count.
6. Fit K-Means, visualise clusters, profile segments, and recommend marketing actions.

> **Important:** Run this notebook with `OnlineRetail.csv` in the same folder. Customer IDs missing from the source cannot be reliably assigned to a customer segment, so those rows are excluded from customer-level RFM analysis. Returns/cancellations and non-positive quantities/prices are excluded from the purchase-value calculations.


In [ ]:
# If needed, install dependencies in your environment:
# %pip install pandas numpy scikit-learn matplotlib seaborn jupyter

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
RANDOM_STATE = 42


## 1. Load and inspect the dataset

In [ ]:
DATA_PATH = "OnlineRetail.csv"
df = pd.read_csv(DATA_PATH, encoding="latin1")

print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
display(df.head())
display(df.info())
display(df.isna().sum().sort_values(ascending=False).to_frame("Missing values"))
display(df.describe(include="all").T)


## 2. Clean and prepare transactions
Convert dates and numeric fields, remove rows without a customer ID/date, remove cancellations/returns and non-positive quantities/prices, and create line-level sales. The cleaning decisions are made explicit so the analysis is reproducible.

In [ ]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["CustomerID"] = pd.to_numeric(df["CustomerID"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")

rows_before = len(df)
missing_customer_rows = df["CustomerID"].isna().sum()

clean = df.dropna(subset=["CustomerID", "InvoiceDate", "Quantity", "UnitPrice"]).copy()
# Invoice numbers beginning with C are cancellations in this dataset.
clean = clean[~clean["InvoiceNo"].astype(str).str.upper().str.startswith("C")]
clean = clean[(clean["Quantity"] > 0) & (clean["UnitPrice"] > 0)]
clean["CustomerID"] = clean["CustomerID"].astype("int64")
clean["Sales"] = clean["Quantity"] * clean["UnitPrice"]

print(f"Original rows: {rows_before:,}")
print(f"Rows missing CustomerID: {missing_customer_rows:,}")
print(f"Rows retained after cleaning: {len(clean):,}")
print(f"Unique customers retained: {clean['CustomerID'].nunique():,}")
print(f"Date range: {clean['InvoiceDate'].min()} to {clean['InvoiceDate'].max()}")
display(clean.head())


## 3. Descriptive statistics and RFM features
- **Recency:** days since the customer's most recent purchase; lower is better.
- **Frequency:** number of distinct invoices/orders; higher is generally better.
- **Monetary:** total net positive sales value in the available dataset; higher indicates greater historical spend.

Customer lifetime value (CLV) is not directly provided by this dataset. Here, **historical customer revenue** (`Monetary`) is used as a practical proxy, not a prediction of future lifetime value. Average purchase value is calculated as total sales divided by distinct invoices.

In [ ]:
reference_date = clean["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = (
    clean.groupby("CustomerID")
    .agg(
        Recency=("InvoiceDate", lambda dates: (reference_date - dates.max()).days),
        Frequency=("InvoiceNo", "nunique"),
        Monetary=("Sales", "sum"),
        TotalItems=("Quantity", "sum"),
        Orders=("InvoiceNo", "nunique"),
    )
    .reset_index()
)

# Average purchase value = customer's historical revenue / number of distinct invoices
rfm["AveragePurchaseValue"] = rfm["Monetary"] / rfm["Frequency"]
rfm["HistoricalRevenueProxy"] = rfm["Monetary"]

print(f"Customer-level RFM records: {len(rfm):,}")
display(rfm[["Recency", "Frequency", "Monetary", "AveragePurchaseValue", "HistoricalRevenueProxy"]].describe().T)
display(rfm.head())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(rfm["Recency"], bins=30, ax=axes[0])
axes[0].set_title("Recency distribution")
sns.histplot(rfm["Frequency"], bins=30, ax=axes[1])
axes[1].set_title("Purchase frequency distribution")
sns.histplot(rfm["Monetary"], bins=40, ax=axes[2])
axes[2].set_title("Historical customer spend distribution")
axes[2].set_xlim(0, rfm["Monetary"].quantile(0.99))
plt.tight_layout()
plt.show()


## 4. Select features and standardise
Use Recency, Frequency, and Monetary (RFM). Frequency and Monetary are typically right-skewed, so `log1p` reduces the influence of extreme values. K-Means is distance-based, so standardisation is important. The cluster profiles later use the original, interpretable units.

In [ ]:
FEATURES = ["Recency", "Frequency", "Monetary"]
X = rfm[FEATURES].copy()
X_model = X.copy()
X_model["Frequency"] = np.log1p(X_model["Frequency"])
X_model["Monetary"] = np.log1p(X_model["Monetary"])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_model)

print("Features used:", FEATURES)
print("Scaled matrix shape:", X_scaled.shape)


## 5. Determine K: Elbow Method and silhouette score
The elbow chart shows how within-cluster sum of squares (inertia) decreases as K increases. The silhouette score provides a second, complementary measure of separation. Select a K that balances separation, interpretability, and useful segment sizes; the highest silhouette alone is not an automatic guarantee of business usefulness.

In [ ]:
k_values = range(2, 9)
inertias, silhouettes = [], []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = model.fit_predict(X_scaled)
    inertias.append(model.inertia_)
    # Sample for speed on larger datasets; reproducible via random_state.
    silhouettes.append(silhouette_score(
        X_scaled, labels,
        sample_size=min(2000, len(X_scaled)),
        random_state=RANDOM_STATE
    ))

evaluation = pd.DataFrame({
    "K": list(k_values),
    "Inertia": inertias,
    "Silhouette score (sampled)": silhouettes
})
display(evaluation)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(evaluation["K"], evaluation["Inertia"], marker="o")
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("Number of clusters (K)")
axes[0].set_ylabel("Inertia")
axes[1].plot(evaluation["K"], evaluation["Silhouette score (sampled)"], marker="o")
axes[1].set_title("Silhouette score by K")
axes[1].set_xlabel("Number of clusters (K)")
axes[1].set_ylabel("Silhouette score")
plt.tight_layout()
plt.show()


### Choose the final K
Review both plots above. The starter value below is **K = 3**, a practical choice for interpretable high-value/active, regular/moderate, and at-risk/low-activity groups in this dataset. If your elbow/silhouette results or business needs suggest another value, change `FINAL_K` and rerun the remaining cells.

In [ ]:
FINAL_K = 3

kmeans = KMeans(n_clusters=FINAL_K, random_state=RANDOM_STATE, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(X_scaled)

print(f"Final K: {FINAL_K}")
print("Silhouette score (sampled):",
      round(silhouette_score(X_scaled, rfm["Cluster"],
                             sample_size=min(2000, len(X_scaled)),
                             random_state=RANDOM_STATE), 3))


## 6. Visualise clusters (two feature combinations)
Scatter plots use original units for interpretability. Monetary is capped at the 99th percentile in the first chart to make the bulk of customers visible; the underlying data is not changed.

In [ ]:
plot_df = rfm.copy()
monetary_cap = plot_df["Monetary"].quantile(0.99)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.scatterplot(data=plot_df, x="Recency", y="Monetary", hue="Cluster",
                palette="deep", alpha=0.65, ax=axes[0])
axes[0].set_ylim(0, monetary_cap)
axes[0].set_title("Customer segments: Recency vs Monetary (99th percentile view)")

sns.scatterplot(data=plot_df, x="Frequency", y="Monetary", hue="Cluster",
                palette="deep", alpha=0.65, ax=axes[1])
axes[1].set_ylim(0, monetary_cap)
axes[1].set_title("Customer segments: Frequency vs Monetary (99th percentile view)")
plt.tight_layout()
plt.show()


## 7. Profile each cluster
Cluster labels are arbitrary numbers; interpret them using the actual averages/medians. The profile includes customer count, average recency, frequency, monetary value, and average order value.

In [ ]:
cluster_profile = (
    rfm.groupby("Cluster")
    .agg(
        Customers=("CustomerID", "count"),
        AvgRecencyDays=("Recency", "mean"),
        MedianRecencyDays=("Recency", "median"),
        AvgOrders=("Frequency", "mean"),
        MedianOrders=("Frequency", "median"),
        AvgMonetary=("Monetary", "mean"),
        MedianMonetary=("Monetary", "median"),
        AvgPurchaseValue=("AveragePurchaseValue", "mean"),
    )
    .round(2)
    .sort_values("AvgMonetary", ascending=False)
)
display(cluster_profile)


In [ ]:
cluster_counts = rfm["Cluster"].value_counts().sort_index()
plt.figure(figsize=(8, 4.5))
ax = sns.barplot(x=cluster_counts.index.astype(str), y=cluster_counts.values)
ax.set_title("Number of customers per cluster")
ax.set_xlabel("Cluster")
ax.set_ylabel("Number of customers")
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f")
plt.tight_layout()
plt.show()


## 8. Segment interpretation and marketing recommendations
The rule-based names below are generated from each cluster's profile. Review the profile table before presenting results, because cluster numbers can change if the dataset or K changes.

In [ ]:
# Assign readable segment names based on profile ranking, not the raw cluster ID.
profile_for_naming = rfm.groupby("Cluster").agg(
    recency=("Recency", "mean"),
    frequency=("Frequency", "mean"),
    monetary=("Monetary", "mean"),
)

# Rank by historical spend and activity; then use recency to distinguish dormant customers.
spend_rank = profile_for_naming["monetary"].rank(method="first", ascending=False)
segment_names = {}
for cluster_id, row in profile_for_naming.iterrows():
    if spend_rank.loc[cluster_id] == 1:
        name = "High-value loyalists"
    elif row["recency"] >= profile_for_naming["recency"].median():
        name = "At-risk / dormant customers"
    else:
        name = "Regular / developing customers"
    segment_names[cluster_id] = name

rfm["Segment"] = rfm["Cluster"].map(segment_names)
segment_summary = (
    rfm.groupby(["Cluster", "Segment"])
    .agg(
        Customers=("CustomerID", "count"),
        AvgRecencyDays=("Recency", "mean"),
        AvgOrders=("Frequency", "mean"),
        AvgHistoricalSpend=("Monetary", "mean"),
        AvgOrderValue=("AveragePurchaseValue", "mean"),
    )
    .round(2)
    .reset_index()
)
display(segment_summary)


In [ ]:
recommendations = {
    "High-value loyalists": (
        "Protect retention: VIP benefits, early access, loyalty rewards, personalised cross-sell, "
        "and proactive service. Avoid unnecessary blanket discounts that erode margin."
    ),
    "Regular / developing customers": (
        "Grow engagement: product recommendations based on prior baskets, bundles, threshold-based "
        "free shipping, and loyalty nudges to increase repeat purchase and basket size."
    ),
    "At-risk / dormant customers": (
        "Win back selectively: targeted reactivation email, relevant new-arrival reminders, and a "
        "time-limited incentive only where expected margin justifies it. Suppress repeatedly "
        "unresponsive customers to control campaign cost."
    ),
}

for _, row in segment_summary.iterrows():
    print(f"Cluster {int(row['Cluster'])} — {row['Segment']}")
    print(f"  Customers: {int(row['Customers']):,} ({row['Customers']/len(rfm):.1%})")
    print(f"  Average recency: {row['AvgRecencyDays']:.1f} days | "
          f"Average orders: {row['AvgOrders']:.2f} | "
          f"Average historical spend: {row['AvgHistoricalSpend']:,.2f}")
    print("  Marketing action:", recommendations.get(row["Segment"],
          "Review cluster metrics and design a segment-specific campaign."))
    print()


## 9. Key business insights and caveats
Use the actual outputs generated above to write the final conclusions. A suitable conclusion should include:

- Which cluster has the highest historical spend and purchase frequency?
- Which cluster has the longest recency (most at risk of churn)?
- What share of customers belongs to each cluster?
- Which campaign should be tested first, and what KPI will measure success (repeat purchase rate, revenue per recipient, redemption rate, or incremental gross margin)?

**Limitations:** This is historical, transaction-based segmentation rather than predictive CLV. The dataset's observation window may not represent a customer's full lifetime. RFM does not include product preferences, customer acquisition cost, campaign response, or profit margin. Validate segments over time and run controlled campaign tests before scaling marketing spend.

In [ ]:
# Compact export of customer-level segments and cluster profile for reporting
rfm.to_csv("customer_segments_rfm.csv", index=False)
cluster_profile.to_csv("cluster_profile_summary.csv")
print("Saved: customer_segments_rfm.csv")
print("Saved: cluster_profile_summary.csv")
